<div style="text-align: justify; max-width: 90ch">

# Lecture 01b: Choosing a model, requirements first

**Status: Mandatory reading.**

In `lec_01a` you ran `qwen3:1.7b` because the course chose it for you. How was it chosen? Not by picking the biggest model that fits a laptop.
A model is a component of a system that has a job, and the job comes with **requirements**: what it must do, in which languages, under which license, at what speed, on which hardware, with what privacy.
Size and hardware fit come *after* that list, never before it.

**What this notebook covers**

- Writing the requirements down before looking at any model.
- Searching the **Hugging Face Hub** from Python and reading a **model card**: license, languages, base model.
- **Quantization**: why one model comes in twenty file sizes, and what you lose.
- Checking hardware fit with `llmfit`.
- Pulling a model from the Hub into Ollama and comparing two candidates on the same prompts.
- Writing a short **decision record**, the habit that survives this course.

</div>

In [1]:
import json
import subprocess
import time

import ollama
import pandas as pd
from huggingface_hub import HfApi, ModelCard

In [2]:
# --- Course configuration: edit TIER only; settings in .env (01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

import os
from pathlib import Path

from dotenv import load_dotenv

REPO = Path.cwd().resolve().parents[1]  # this notebook runs from reading_material/
load_dotenv(REPO / ".env")  # your settings file: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
os.environ.setdefault(
    "HF_HUB_DISABLE_PROGRESS_BARS", "1"
)  # no download bars in saved outputs
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-01-basics"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. The wrong question and the right one

"Which model is best?" has no answer. "Which model is best *for this job*?" does, once the job is written down. Here is the table for this course's own system,
an assistant that answers questions about the Python course notes on a student laptop:

| Requirement | For this course | Why it matters |
| --- | --- | --- |
| **Task** | answer questions from retrieved notes (lecture 2); call tools (lecture 3) | not every small model follows instructions or calls tools |
| **Languages** | English notes; Greek questions must work | the tokenizer and training data decide this |
| **License** | free for any use, including commercial, no naming clauses | a product built on it must be shippable |
| **Context** | at least 8 000 tokens (notes + question + answer) | lecture 2 puts several pages into one prompt |
| **Latency** | interactive on a CPU laptop: 10+ tokens/s | a 60-second answer is not a chat |
| **Hardware and cost** | 8-16 GB RAM, no GPU, no API bill | who the students are |
| **Privacy** | everything stays on the laptop | course notes are harmless; a company's documents are not |
| **Maintenance** | one active family with several sizes | the `cpu` and `gpu` tiers should share one code base |

```text
WRONG: "the biggest model that fits in my RAM"
RIGHT: "the smallest model that meets every requirement, then the biggest of those that fits"
```

The rest of this notebook checks candidates against that table.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Where models live: the Hugging Face Hub

The [Hugging Face Hub](https://huggingface.co/models) is to models what GitHub is to code: hundreds of thousands of repositories, each holding weights, a tokenizer, and a README called the **model card**.
You used it in the Python course to host a Gradio app; today you query it from Python.

Public information needs no login, but anonymous requests are rate-limited and some models require accepting a license first.
Guide `01d_env_hugging_face` showed how to create a **read token** and put it in `.env`. The configuration cell already copied `.env` into environment variables (`lec_01a`,
section 0), and `huggingface_hub` reads `HF_TOKEN` from there. The token never appears in the notebook, so it cannot end up on GitHub.

</div>

In [3]:
token = os.environ.get("HF_TOKEN")
print("HF_TOKEN loaded:", bool(token), "" if not token else f"({token[:6]}...)")

HF_TOKEN loaded: False 


<div style="text-align: justify; max-width: 90ch">

Start from the **languages** requirement. Model cards carry language tags; `el` is Greek. `pipeline_tag` narrows to models that generate text, and `sort="downloads"` puts the popular ones first.

</div>

In [4]:
api = HfApi()

greek_models = api.list_models(
    pipeline_tag="text-generation", filter="el", sort="downloads", limit=10
)
pd.DataFrame(
    [{"model": m.id, "downloads": m.downloads, "likes": m.likes} for m in greek_models]
)

,model,downloads,likes
0,speakleash/Bielik-11B-v3.0-Instruct-awq,218127,3
1,utter-project/EuroLLM-22B-Instruct-2512,188020,82
2,facebook/xglm-564M,112944,54
3,utter-project/EuroLLM-9B-Instruct-2512,30261,14
4,BSC-LT/salamandra-7b-instruct,29727,84
5,utter-project/EuroLLM-9B-Instruct,20260,219
6,CohereLabs/tiny-aya-base,18447,65
7,CohereLabs/aya-expanse-8b,15404,450
8,utter-project/EuroLLM-1.7B-Instruct,14146,103
9,ai-forever/mGPT,13989,271


<div style="text-align: justify; max-width: 90ch">

Read the names: they encode the **family** (EuroLLM, Salamandra, Bielik), the **size** (`9B`, `7b`), the **variant** (`Instruct` means fine-tuned to follow instructions,
which chat needs), and sometimes the **format** (`GGUF`, `awq` are compressed files, next section).

Search results change every month; this list will look different when you run it. The requirements table does not change, and that is the point of writing it first.

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. Reading a model card

A model card is a README with a metadata header: `license`, `language`, `base_model`, `pipeline_tag`. Three candidates from three companies, side by side.
`ModelCard.load` fetches only the README, not the weights.

</div>

In [5]:
candidates = [
    "Qwen/Qwen3-1.7B",
    "meta-llama/Llama-3.2-1B-Instruct",
    "google/gemma-3-1b-it",
]

rows = []
for repo in candidates:
    data = ModelCard.load(repo).data.to_dict()
    rows.append(
        {
            "model": repo,
            "license": data.get("license"),
            "languages (metadata)": data.get("language"),
            "base_model": data.get("base_model"),
        }
    )
pd.DataFrame(rows)

,model,license,languages (metadata),base_model
0,Qwen/Qwen3-1.7B,apache-2.0,None,[Qwen/Qwen3-1.7B-Base]
1,meta-llama/Llama-3.2-1B-Instruct,llama3.2,"[en, de, fr, it, pt, hi, es, th]",None
2,google/gemma-3-1b-it,gemma,None,google/gemma-3-1b-pt


<div style="text-align: justify; max-width: 90ch">

Against the requirements table:

- **Llama 3.2** lists eight languages and Greek is not one of them: it fails the *languages* requirement before we look at anything else.
  Its license, `llama3.2`, is a custom agreement with conditions on naming and on very large services: a legal review, not a checkbox.
- **Gemma 3** ships under Google's own `gemma` terms, also custom.
- **Qwen3** is `apache-2.0`, the same permissive license as most Python libraries you use, and metadata says nothing about languages.

When metadata is missing, read the card text.

</div>

In [6]:
qwen_card = ModelCard.load("Qwen/Qwen3-1.7B")

claims = [
    line.strip() for line in qwen_card.text.splitlines() if "languages" in line.lower()
]
for claim in claims[:3]:
    print(claim[:160])

- **Support of 100+ languages and dialects** with strong capabilities for **multilingual instruction following** and **translation**.


<div style="text-align: justify; max-width: 90ch">

"Support of 100+ languages" is a claim by the vendor, not a measurement. Section 6 tests it on a Greek question, because a requirement is met when *you* have checked it.

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. Size and quantization

Memory is arithmetic: **parameters × bytes per parameter**. A model trained in 16-bit floats stores each weight in 2 bytes.
**Quantization** rounds the weights to fewer bits after training; the file shrinks, the model gets a little worse.

| Format | Bytes per weight | `qwen3:1.7b` on disk | Typical quality loss |
| --- | --- | --- | --- |
| `BF16` / `FP16` | 2 | 3.4 GB | none (the original) |
| `Q8_0` | 1 | 1.8 GB | negligible |
| `Q4_K_M` | about 0.6 | 1.1 GB | small; the usual default |
| `Q2_K` | about 0.4 | 0.8 GB | noticeable |

**GGUF** is the file format that Ollama (through the `llama.cpp` engine) runs. Community accounts publish the same model in every quantization;
the Hub API lists the files with their sizes.

</div>

In [7]:
gguf_repo = "unsloth/Qwen3-1.7B-GGUF"
info = api.model_info(gguf_repo, files_metadata=True)

gguf_files = pd.DataFrame(
    [
        {"file": s.rfilename, "size_gb": round(s.size / 1e9, 2)}
        for s in info.siblings
        if s.rfilename.endswith(".gguf")
    ]
)
gguf_files["quantization"] = gguf_files["file"].str.extract(r"Qwen3-1\.7B-(.+)\.gguf")
gguf_files.sort_values("size_gb")

,file,size_gb,quantization
16,Qwen3-1.7B-UD-IQ1_S.gguf,0.54,UD-IQ1_S
15,Qwen3-1.7B-UD-IQ1_M.gguf,0.56,UD-IQ1_M
18,Qwen3-1.7B-UD-IQ2_XXS.gguf,0.61,UD-IQ2_XXS
17,Qwen3-1.7B-UD-IQ2_M.gguf,0.71,UD-IQ2_M
19,Qwen3-1.7B-UD-IQ3_XXS.gguf,0.77,UD-IQ3_XXS
3,Qwen3-1.7B-Q2_K.gguf,0.78,Q2_K
4,Qwen3-1.7B-Q2_K_L.gguf,0.78,Q2_K_L
20,Qwen3-1.7B-UD-Q2_K_XL.gguf,0.80,UD-Q2_K_XL
6,Qwen3-1.7B-Q3_K_S.gguf,0.87,Q3_K_S
5,Qwen3-1.7B-Q3_K_M.gguf,0.94,Q3_K_M


<div style="text-align: justify; max-width: 90ch">

The 4-bit file is a third of the original and is what `ollama pull qwen3:1.7b` gave you in `lec_01a` (its quantization column said `Q4_K_M`).
Add the memory for the context window (a few hundred MB at 8 000 tokens) and you have the RAM a model needs to run.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. Does it fit? Ask llmfit

`llmfit` measures your RAM, VRAM and CPU and scores a catalog of models on **fit**, **speed**, **quality** and **context**.
In a terminal you run `uvx llmfit recommend`; here we call the same command from Python and keep the columns that map to our requirements.

</div>

In [8]:
def llmfit(*args):
    """Run `uvx llmfit <args> --json` and return the list of model records."""
    result = subprocess.run(
        ["uvx", "llmfit", *args, "--json"], capture_output=True, text=True, timeout=180
    )
    return json.loads(result.stdout)["models"]


COLUMNS = [
    "name",
    "params_b",
    "license",
    "memory_required_gb",
    "fit_label",
    "estimated_tps",
    "run_mode",
    "effective_context_length",
]

recommended = pd.DataFrame(llmfit("recommend"))[COLUMNS]
recommended.head(5)

,name,params_b,license,memory_required_gb,fit_label,estimated_tps,run_mode,effective_context_length
0,TelperionAI/Huihui-Qwen3.8-27B-abliterated-INT...,7.84,apache-2.0,4.48,Perfect,37.0,GPU,4096
1,TelperionAI/Qwen3.8-27B-INT4-AWQ-GPTQ-gdn4,7.30,apache-2.0,4.21,Perfect,39.8,GPU,4096
2,TelperionAI/Qwen3.8-27B-INT4-AWQ-GPTQ,7.84,apache-2.0,4.48,Perfect,37.0,GPU,4096
3,Qwen/Qwen2.5-Coder-7B-Instruct-GPTQ-Int4,7.62,apache-2.0,4.75,Perfect,38.1,GPU,8192
4,Orion-zhen/Qwen2.5-Coder-7B-Instruct-AWQ,7.62,apache-2.0,4.75,Perfect,38.1,GPU,8192


<div style="text-align: justify; max-width: 90ch">

Useful, and also a reminder that a catalog is not a requirements table: the top rows are whatever scores best on *this* hardware, including variants nobody should ship to customers.
The question we need answered is narrower: do our two tier candidates fit? `llmfit plan` answers it for the file Ollama actually ships (`Q4_K_M`,
section 4) at the 8 000-token context of the requirements table, with a speed estimate for each way to run it: on the GPU, split between GPU and CPU, or on the CPU only.

</div>

In [9]:
def llmfit_plan(model):
    """Run `uvx llmfit plan` for Ollama's Q4_K_M build and return one row per model."""
    result = subprocess.run(
        [
            "uvx",
            "llmfit",
            "plan",
            model,
            "--quant",
            "Q4_K_M",
            "--context",
            "8192",
            "--json",
        ],
        capture_output=True,
        text=True,
        timeout=180,
    )
    plan = json.loads(result.stdout)
    speeds = {path["path"]: path["estimated_tps"] for path in plan["run_paths"]}
    return {
        "name": plan["model_name"],
        "min_vram_gb": plan["minimum"]["vram_gb"],
        "min_ram_gb": plan["minimum"]["ram_gb"],
        "gpu_tps": speeds["gpu"],
        "cpu_only_tps": speeds["cpu_only"],
        "run_mode_here": plan["current"]["run_mode"],
    }


tiers = pd.DataFrame([llmfit_plan("Qwen/Qwen3-1.7B"), llmfit_plan("Qwen/Qwen3-8B")])
tiers.round(1)

,name,min_vram_gb,min_ram_gb,gpu_tps,cpu_only_tps,run_mode_here
0,Qwen/Qwen3-1.7B,2.6,8.0,124.5,11.9,Gpu
1,Qwen/Qwen3-8B,6.4,8.0,30.9,2.9,CpuOffload


<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

Compare the two rows with your neighbours: `cpu_only_tps` of the 8B row is a speed nobody would call interactive, and `min_vram_gb` is the GPU memory the `gpu` tier needs.
`run_mode_here` is how llmfit would run each model on *this* laptop. That is the whole `TIER` decision, made from a measurement.
Notice what llmfit does **not** know: your languages, your privacy rules, your task. It answers "can I run it?", the requirements table answers "should I?".

</div>

<div style="text-align: justify; max-width: 90ch">

## 6. Pull a candidate and compare

Two candidates meet the requirements on paper: `qwen3:1.7b` and its smaller sibling `qwen3:0.6b`, which would be faster on a weak laptop.
Both are in Ollama's library:

```bash
ollama pull qwen3:0.6b      # about 520 MB, one-time
```

</div>

In [10]:
SMALL_MODEL = "qwen3:0.6b"
check_ollama(OLLAMA_HOST, [CHAT_MODEL, SMALL_MODEL])

client = ollama.Client(host=OLLAMA_HOST)

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b, qwen3:0.6b


<div style="text-align: justify; max-width: 90ch">

Same three prompts to both models: an English fact from the course, a Greek question, and a small reasoning trap. Thinking stays off, temperature stays at 0, so the only difference is the model.

</div>

In [11]:
prompts = {
    "english": "In one sentence, what does train_test_split do in scikit-learn?",
    "greek": "Σε μία πρόταση, τι κάνει η συνάρτηση train_test_split του scikit-learn; Απάντησε στα ελληνικά.",
    "reasoning": "Which number is larger, 9.11 or 9.9? Answer with the number only.",
}


def compare(models, prompts):
    """Ask every prompt to every model; return one row per answer with its speed."""
    rows = []
    for model in models:
        for name, prompt in prompts.items():
            response = client.chat(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                think=False,
                options={"temperature": 0, "num_predict": 80},
            )
            rows.append(
                {
                    "model": model,
                    "prompt": name,
                    "tok_per_s": round(
                        response.eval_count / (response.eval_duration / 1e9), 1
                    ),
                    "answer": response.message.content.strip().replace("\n", " ")[:110],
                }
            )
    return pd.DataFrame(rows)


compare([SMALL_MODEL, CHAT_MODEL], prompts)

,model,prompt,tok_per_s,answer
0,qwen3:0.6b,english,224.5,Train_test_split() is a function in scikit-lea...
1,qwen3:0.6b,greek,227.1,Συνάρτηση `train_test_split` του scikit-learn ...
2,qwen3:0.6b,reasoning,241.5,The number **9.9** is larger than **9.11**.
3,qwen3:1.7b,english,108.4,The `train_test_split` function in scikit-lear...
4,qwen3:1.7b,greek,106.6,Η συνάρτηση `train_test_split` του scikit-lear...
5,qwen3:1.7b,reasoning,147.6,9.9


<div style="text-align: justify; max-width: 90ch">

Read the table against the requirements, not against your taste:

- **Latency**: the small model is faster (tokens/s), as expected from a third of the parameters.
- **Languages**: compare the Greek answers. Is the small one grammatical and correct, or does it drift into English or nonsense?
- **Task**: the reasoning trap tells you whether a model *reads* or *pattern-matches*.

A cheaper model that fails a requirement is not cheaper: it is unusable.

</div>

<div style="text-align: justify; max-width: 90ch">

### 6b. Straight from the Hub

⏱ **Skip if running long.** Nothing below depends on it.

Ollama's library is curated; the Hub has everything. Ollama can pull a GGUF file directly from a Hub repository, named `hf.co/<account>/<repository>:<quantization>`:

```bash
ollama pull hf.co/unsloth/Qwen3-0.6B-GGUF:Q8_0      # about 640 MB
```

The catch: a Hub file brings its own **chat template**, the recipe that turns your messages into the model's input, and Ollama's `think` switch only works with templates it knows.
Watch what the same prompt returns.

</div>

In [12]:
HUB_MODEL = "hf.co/unsloth/Qwen3-0.6B-GGUF:Q8_0"
pulled = {m.model for m in client.list().models}

if HUB_MODEL in pulled:
    response = client.chat(
        model=HUB_MODEL,
        messages=[{"role": "user", "content": prompts["reasoning"]}],
        think=False,
        options={"temperature": 0, "num_predict": 60},
    )
    print(response.message.content[:200])
else:
    print(f"Not pulled; run: ollama pull {HUB_MODEL}")

<think>
Okay, so I need to figure out which number is larger between 9.11 and 9.9. Let me think. Both numbers are in decimal form, right? 9.11 and 9.9. Hmm. Let me write them down to compare


<div style="text-align: justify; max-width: 90ch">

The model reasons inside `<think>` tags *in the answer text*, and `think=False` changed nothing: the template did not expose the switch.
Same weights, different packaging, different behaviour. For the course we stay with the library build; the Hub route is there when the library does not have the model you need, and the requirements table gets one more row:
**"does the packaging work with my tooling?"**

</div>

<div style="text-align: justify; max-width: 90ch">

## 7. What thinking costs

⏱ **Skip if running long.** The lesson is the number; nothing below depends on it.

`qwen3` models can reason before answering (`think=True`). The reasoning is generated as tokens, so it costs time in proportion.
Try the trap again, with thinking on.

</div>

In [13]:
started = time.time()
response = client.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": prompts["reasoning"]}],
    think=True,
    options={"temperature": 0, "num_predict": 1500},
)
elapsed = time.time() - started

print(f"{elapsed:.1f} s, {response.eval_count} tokens generated")
print("thinking (first 300 chars):", (response.message.thinking or "")[:300])
print("answer:", response.message.content.strip())

9.5 s, 930 tokens generated
thinking (first 300 chars): Okay, let's see. The question is asking which number is larger between 9.11 and 9.9. Hmm. Both numbers are decimals, right? So, I need to compare them step by step.

First, let me write them down again to visualize better: 9.11 and 9.9. Both have the same integer part, which is 9. So, the whole numb
answer: To determine which number is larger between 9.11 and 9.9, we compare their decimal parts after the integer part.

Both numbers have the same integer part: **9**. Therefore, we focus on the decimal parts:

- 9.11 has a decimal part of **0.11**
- 9.9 has a decimal part of **0.9**

Since **0.9 > 0.11**, the number **9.9** is larger than **9.11**.

---

**Final Answer:**  
$$
\boxed{9.9}
$$


<div style="text-align: justify; max-width: 90ch">

Most of those tokens were the model talking to itself. Reasoning is a capability you switch on where a decision is worth the wait (lecture 3's agents) and off everywhere else.
That is also a requirements question: "how long may an answer take?"

</div>

<div style="text-align: justify; max-width: 90ch">

## 8. The decision record

Five lines, written once, kept next to the code. It is what a colleague, or you in six months, will ask for.

> **Decision:** `qwen3:1.7b` (Q4_K_M) for the `cpu` tier, `qwen3:8b` for the `gpu` tier.
> **Requirements:** RAG over English notes with Greek questions; Apache-2.0 or similar; 8k+ context; 10+ tok/s on a CPU laptop; local only; one family for both tiers.
> **Candidates checked:** Llama 3.2 (no Greek, custom license), Gemma 3 (custom terms), Gemma 4 (Apache-2.0, but its smallest Ollama build is 7.2 GB), Qwen3.5 (Apache-2.0, 201 languages; the successor to test next edition), Qwen3-0.6B (fast, but check its Greek and its reasoning in section 6), Qwen3-1.7B (meets all), Qwen3-8B (GPU only).
> **Evidence:** model cards (section 3), Greek and reasoning prompts (section 6), llmfit fit (section 5), and the candidate table in guide `01b_llmfit` (`llmfit plan` at Ollama's `Q4_K_M`).
> **Revisit when:** a newer family in the same license class appears, or a requirement changes (for example Greek notes).

In the exercises you write one for a different scenario; in the final assignment you write one for your own.

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- Write the **requirements table** first: task, languages, license, context, latency, hardware, privacy, maintenance.
- The **Hugging Face Hub** is searchable from Python; model cards carry the license and language metadata, and their text carries the claims.
- **Quantization** trades bytes for a little quality; `Q4_K_M` is the usual laptop default; memory is parameters × bytes.
- **llmfit** answers "can I run it?"; the table answers "should I?".
- Ollama pulls models from its library or straight from the Hub (`hf.co/...`); compare candidates on *your* prompts.
- Reasoning costs tokens, so time; switch it on only where a decision is worth it.
- Keep a five-line **decision record**.

**Next: `lec_01c`**, prompts as code. The model is chosen; now we make what we send it reproducible.

</div>